# 03 DP-CTGAN Synthesis and Fairness Auditing

In this notebook, we'll demonstrate how to synthesize a Differential Privacy (DP) audit table using `DPCTGANSynthesizer`, and then use it to audit both a pre-trained `Logistic Regression` and `FT-Transformer` model for fairness.

> **Note**: This notebook expects the models to have already been trained by `01_train_models_step_by_step.ipynb`. If they are not found, it will raise an error asking you to run it first.

In [1]:
import sys
import os
from pathlib import Path

# Add src directory to path
sys.path.append(str(Path(".").resolve().parent / "src"))

import numpy as np
import pandas as pd
import joblib

from fairness_auditbench.datasets.folktables_acs import ACSPublicCoverageDataset
from fairness_auditbench.synthesizers.registry import get_synthesizer
import fairness_auditbench.synthesizers.private_pgm  # Ensure it registers
from fairness_auditbench.metrics.fairness import compute_fairness_metrics
from fairness_auditbench.runners.audit import evaluate_model

## 1. Data Preparation
We use the full dataset corresponding to the trained model (default CA, 2018).

In [2]:
dataset = ACSPublicCoverageDataset(
    states=["CA"], 
    year=2018, 
    fast_dev_run=False
)

train_df, val_df, test_df, spec = dataset.get_splits(seed=0)
print(f"Test set size: {len(test_df)}")

Test set size: 27712


## 2. Check for Pre-trained Models
Before synthesizing data, we verify that the models trained in Notebook 01 actually exist.

In [3]:
# This matches the output directory of train_model.py for the full dataset
model_dir_logreg = Path("../results/models/acs_public_coverage/logreg/seed=0")
model_dir_ft = Path("../results/models/acs_public_coverage/ft_transformer/seed=0")

for m_dir in [model_dir_logreg, model_dir_ft]:
    if not m_dir.exists():
        raise FileNotFoundError(
            f"Model not found at {m_dir}. "
            "Please run Notebook 01_train_models_step_by_step.ipynb first to train the model!"
        )
print("Found pre-trained Logistic Regression and FT-Transformer models.")

Found pre-trained Logistic Regression and FT-Transformer models.


## 3. Synthesize Audit Table with DP-CTGAN
Using the DP synthesizer to create a synthetic version of the  split. We run it via .

In [4]:
epsilon = 1.0
delta = 1e-6
seed = 0    

synth = get_synthesizer(
    'dpctgan',
    epochs=500,               # Increased training time for better convergence
)
synth.fit(test_df, spec=spec, epsilon=epsilon, delta=delta, seed=seed)
synth_df = synth.sample(n=len(test_df), seed=seed)

print("Synthetic data generated successfully.")

Spent 0.1 epsilon on preprocessor, leaving 0.9 for training


/tmp/python-venv/fairness-auditbench_venv/lib/python3.11/site-packages/opacus/privacy_engine.py:638: UserWarning: The sample rate will be defined from ``batch_size`` and ``sample_size``.The returned privacy budget will be incorrect.
  warnings.warn(
/tmp/python-venv/fairness-auditbench_venv/lib/python3.11/site-packages/opacus/privacy_engine.py:229: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_rng`` turned on.
  warnings.warn(
/tmp/python-venv/fairness-auditbench_venv/lib/python3.11/site-packages/torch/nn/modules/module.py:1866: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_back

Epoch 1, Loss G: 0.7028, Loss D: 1.3891
epsilon is 0.14000572045311915, alpha is 63.0
Epoch 2, Loss G: 0.7050, Loss D: 1.3860
epsilon is 0.18822882627500787, alpha is 63.0
Epoch 3, Loss G: 0.6949, Loss D: 1.3894
epsilon is 0.23645193209689658, alpha is 63.0
Epoch 4, Loss G: 0.7106, Loss D: 1.3957
epsilon is 0.28467503791878535, alpha is 63.0
Epoch 5, Loss G: 0.7206, Loss D: 1.3900
epsilon is 0.33050001676955953, alpha is 56.0
Epoch 6, Loss G: 0.7212, Loss D: 1.3891
epsilon is 0.3712268169088867, alpha is 51.0
Epoch 7, Loss G: 0.7205, Loss D: 1.3957
epsilon is 0.40829656845548773, alpha is 47.0
Epoch 8, Loss G: 0.7210, Loss D: 1.3883
epsilon is 0.4425747994915673, alpha is 44.0
Epoch 9, Loss G: 0.7273, Loss D: 1.3848
epsilon is 0.4746450190373351, alpha is 41.0
Epoch 10, Loss G: 0.7223, Loss D: 1.3975
epsilon is 0.504869915921788, alpha is 39.0
Epoch 11, Loss G: 0.7413, Loss D: 1.3820
epsilon is 0.5335987019162898, alpha is 37.0
Epoch 12, Loss G: 0.7293, Loss D: 1.3923
epsilon is 0.5609

## 4. Audit Fairness (Logistic Regression)
Load our trained Logistic Regression model and compute predictions on both real and synthetic data.

In [5]:
# Real Predictions
real_preds_df_logreg = evaluate_model(test_df, "logreg", model_dir_logreg, spec)
real_metrics_logreg = compute_fairness_metrics(
    real_preds_df_logreg, spec.label_col, "_y_pred", spec.sensitive_col
)

# Synthetic Predictions
synth_preds_df_logreg = evaluate_model(synth_df, "logreg", model_dir_logreg, spec)
synth_metrics_logreg = compute_fairness_metrics(
    synth_preds_df_logreg, spec.label_col, "_y_pred", spec.sensitive_col
)

## 5. Audit Fairness (FT-Transformer)
Load our trained FT-Transformer model and compute predictions.

In [6]:
# Real Predictions
real_preds_df_ft = evaluate_model(test_df, "ft_transformer", model_dir_ft, spec)
real_metrics_ft = compute_fairness_metrics(
    real_preds_df_ft, spec.label_col, "_y_pred", spec.sensitive_col
)

# Synthetic Predictions
synth_preds_df_ft = evaluate_model(synth_df, "ft_transformer", model_dir_ft, spec)
synth_metrics_ft = compute_fairness_metrics(
    synth_preds_df_ft, spec.label_col, "_y_pred", spec.sensitive_col
)

/tmp/python-venv/fairness-auditbench_venv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/tmp/python-venv/fairness-auditbench_venv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


## 6. View Audit Error
We compute the difference between the fairness metric measured on the real data vs the synthetic data.

In [7]:
def print_audit_error(model_name, real_m, synth_m):
    audit_error = {
        k: abs(real_m.get(k, 0.0) - synth_m.get(k, 0.0))
        for k in ["demographic_parity_score", "equal_opportunity_score", "equalized_odds_score"]
    }
    
    print(f"\n{'='*15} {model_name} {'='*15}")
    print("--- REAL METRICS ---")
    for k, v in real_m.items():
        if not isinstance(v, dict):
            print(f"{k}: {v:.4f}")
            
    print("\n--- SYNTH METRICS ---")
    for k, v in synth_m.items():
        if not isinstance(v, dict):
            print(f"{k}: {v:.4f}")
            
    print("\n--- AUDIT ERRORS ---")
    for k, v in audit_error.items():
        print(f"{k}: {v:.4f}")

print_audit_error("Logistic Regression", real_metrics_logreg, synth_metrics_logreg)
print_audit_error("FT-Transformer", real_metrics_ft, synth_metrics_ft)



=============== Logistic Regression ===============
--- REAL METRICS ---
demographic_parity_score: 0.3414
equal_opportunity_score: 0.1359
equalized_odds_score: 0.4194

--- SYNTH METRICS ---
demographic_parity_score: 0.0735
equal_opportunity_score: 0.0450
equalized_odds_score: 0.0811

--- AUDIT ERRORS ---
demographic_parity_score: 0.2678
equal_opportunity_score: 0.0908
equalized_odds_score: 0.3383

=============== FT-Transformer ===============
--- REAL METRICS ---
demographic_parity_score: 0.3313
equal_opportunity_score: 0.3057
equalized_odds_score: 0.4086

--- SYNTH METRICS ---
demographic_parity_score: 0.7130
equal_opportunity_score: 0.2738
equalized_odds_score: 0.6013

--- AUDIT ERRORS ---
demographic_parity_score: 0.3817
equal_opportunity_score: 0.0319
equalized_odds_score: 0.1927
